In [ ]:
# ============================================================
# Extract Chiang Mai District 7 Election Result
# แยกข้อมูลเชียงใหม่ เขต 7 ออกเป็น 2 ตาราง:
# 1) สส. เขต        -> column ที่ขึ้นต้นด้วย "เขต_"
# 2) บัญชีรายชื่อ  -> column ที่ขึ้นต้นด้วย "บช_"
#
# Output:
# - long table ของ สส. เขต
# - long table ของบัญชีรายชื่อ
# - summary รวมคะแนนตามพรรค
# - wide summary: พรรค 1 แถว และแยกคะแนนตาม Subdistrict / ตำบล
#
# หมายเหตุ:
# - ใช้ province_number == 7 เพื่อเลือกเชียงใหม่เขต 7
# - ต้องเป็นจังหวัดเชียงใหม่เท่านั้น
# - รวมคะแนน "ไม่เลือกผู้ใด" เป็นเหมือนตัวเลือกหนึ่ง
# - รวม "บัตรดี" และ "บัตรเสีย" เป็นแถวใน summary ด้วย
# - ไม่เอา "ผู้มีสิทธิ์" และ "ผู้มาใช้สิทธิ์" ไปรวมเป็นคะแนนพรรค
# - แต่เก็บ "ผู้มีสิทธิ์" และ "ผู้มาใช้สิทธิ์" ไว้ในตาราง long ด้วย
# ============================================================


# ============================================================
# 1) Install required libraries
# ============================================================
# ถ้าใช้ใน Colab ให้เปิดบรรทัดนี้
# !pip install gdown openpyxl -q


# ============================================================
# 2) Import libraries
# ============================================================

import re
from pathlib import Path

import pandas as pd
import gdown


# ============================================================
# 3) Config
# ============================================================

DRIVE_URL = "https://drive.google.com/file/d/1inCkmOw8YtzX6kRf5R-yrU3gSRlIh6ZM/view?usp=drive_link"

OUTPUT_FILE = "election_result_file"

DISTRICT_NO = 7

CONSTITUENCY_PREFIX = "เขต_"
PARTYLIST_PREFIX = "บช_"

# ไม่เอาสองช่องนี้ไปคิดเป็นคะแนนพรรค
# แต่ยังเก็บไว้ใน long table เป็น metadata
EXCLUDE_KEYWORDS = [
    "ผู้มีสิทธิ์",
    "ผู้มาใช้สิทธิ์"
]

# column ที่อยากเก็บไว้เป็นข้อมูลประกอบ ถ้าในไฟล์มี column เหล่านี้
ID_COLS_CANDIDATES = [
    "province_number",
    "province",
    "province_name",
    "จังหวัด",
    "district",
    "district_name",
    "เขต",
    "amphoe",
    "อำเภอ",
    "subdistrict",
    "subdistrict_name",
    "tambon",
    "ตำบล",
    "moo",
    "หมู่",
    "precinct_no",
    "unit_no",
    "หน่วย",
    "หน่วยเลือกตั้ง",
    "polling_station",
    "สถานที่เลือกตั้ง",
    "latitude",
    "longitude",
    "lat",
    "lon",
    "lng"
]


# ============================================================
# 4) Download file from Google Drive
# ============================================================

def download_from_google_drive(drive_url, output_file):
    """
    Download file from Google Drive shared link.
    """

    match = re.search(r"/d/([^/]+)", drive_url)

    if not match:
        raise ValueError("Cannot extract file id from Google Drive URL.")

    file_id = match.group(1)
    download_url = f"https://drive.google.com/uc?id={file_id}"

    print("Downloading file from Google Drive...")
    gdown.download(download_url, output_file, quiet=False)

    print(f"Downloaded as: {output_file}")
    return output_file


# ============================================================
# 5) Read file automatically
# ============================================================

def read_election_file(path):
    """
    Read election result file.
    Try Excel first, then CSV with common Thai encodings.
    """

    path = Path(path)

    # Try Excel
    try:
        df = pd.read_excel(path)
        print("File type detected: Excel")
        return df
    except Exception:
        pass

    # Try CSV
    encodings = ["utf-8-sig", "utf-8", "cp874", "tis-620"]

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"File type detected: CSV, encoding={enc}")
            return df
        except Exception:
            continue

    raise ValueError("Cannot read file. Please check file format.")


# ============================================================
# 6) Clean column names
# ============================================================

def clean_column_names(df):
    """
    Clean dataframe column names.
    """

    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace("\t", " ", regex=False)
    )

    return df


# ============================================================
# 7) Helper functions
# ============================================================

def is_vote_col(col, prefix):
    """
    เลือก column คะแนนที่ขึ้นต้นด้วย prefix เช่น เขต_ หรือ บช_

    เอาเข้า:
    - พรรค
    - ไม่เลือกผู้ใด
    - บัตรดี
    - บัตรเสีย

    ไม่เอา:
    - ผู้มีสิทธิ์
    - ผู้มาใช้สิทธิ์
    """

    col = str(col).strip()

    if not col.startswith(prefix):
        return False

    for keyword in EXCLUDE_KEYWORDS:
        if keyword in col:
            return False

    return True


def is_meta_turnout_col(col, prefix):
    """
    เลือก column ผู้มีสิทธิ์ / ผู้มาใช้สิทธิ์ แยกไว้เป็น metadata
    """

    col = str(col).strip()

    if not col.startswith(prefix):
        return False

    return any(keyword in col for keyword in EXCLUDE_KEYWORDS)


def clean_party_name(col, prefix):
    """
    ตัด prefix ออกจากชื่อ column เพื่อให้เหลือแค่ชื่อพรรค

    ตัวอย่าง:
    เขต_เพื่อไทย -> เพื่อไทย
    บช_ก้าวไกล -> ก้าวไกล
    เขต_ไม่เลือกผู้ใด -> ไม่เลือกผู้ใด
    เขต_บัตรดี -> บัตรดี
    เขต_บัตรเสีย -> บัตรเสีย
    """

    return str(col).replace(prefix, "", 1).strip()


def get_available_id_cols(df):
    """
    เลือกเฉพาะ id columns ที่มีอยู่จริงในไฟล์
    """

    available_cols = []

    for col in ID_COLS_CANDIDATES:
        if col in df.columns and col not in available_cols:
            available_cols.append(col)

    return available_cols


def clean_numeric_series(s):
    """
    แปลงคะแนนให้เป็นตัวเลข
    รองรับกรณีมี comma เช่น 1,234
    """

    return (
        s.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": None, "nan": None, "None": None})
        .pipe(pd.to_numeric, errors="coerce")
        .fillna(0)
        .astype(int)
    )


# ============================================================
# 8) Filter Chiang Mai District 7
# ============================================================

def filter_chiangmai_district7(df):
    """
    Filter เฉพาะจังหวัดเชียงใหม่ เขต 7

    เงื่อนไข:
    1) province ต้องเป็น เชียงใหม่
    2) province_number ต้องเท่ากับ 7
    """

    if "province_number" not in df.columns:
        raise ValueError("Column 'province_number' not found in file.")

    df = df.copy()

    # หา column จังหวัดแบบยืดหยุ่น
    province_col = None
    for col in ["province", "province_name", "จังหวัด"]:
        if col in df.columns:
            province_col = col
            break

    if province_col is None:
        raise ValueError(
            "Cannot find province column. Expected one of: province, province_name, จังหวัด"
        )

    province_number_numeric = pd.to_numeric(
        df["province_number"],
        errors="coerce"
    )

    df_cm7 = df[
        df[province_col].astype(str).str.contains("เชียงใหม่", na=False)
        & (province_number_numeric == DISTRICT_NO)
    ].copy()

    return df_cm7


# ============================================================
# 9) Make long vote table
# ============================================================

def make_vote_table(df, prefix, election_type):
    """
    สร้างตารางคะแนนแบบ long format

    Input:
    - df: dataframe ที่ filter เขต 7 แล้ว
    - prefix: "เขต_" หรือ "บช_"
    - election_type: ชื่อประเภทเลือกตั้ง

    Output:
    - df_long: ตาราง long format
    - vote_cols: รายชื่อ column คะแนนที่ถูกใช้
    - meta_turnout_cols: รายชื่อ column ผู้มีสิทธิ์ / ผู้มาใช้สิทธิ์
    """

    df = df.copy()

    id_cols = get_available_id_cols(df)

    vote_cols = [
        col for col in df.columns
        if is_vote_col(col, prefix)
    ]

    meta_turnout_cols = [
        col for col in df.columns
        if is_meta_turnout_col(col, prefix)
    ]

    if len(vote_cols) == 0:
        print(f"Warning: No vote columns found for prefix '{prefix}'")

    keep_cols = id_cols + meta_turnout_cols + vote_cols

    # กัน column ซ้ำ
    keep_cols = list(dict.fromkeys(keep_cols))

    df_part = df[keep_cols].copy()

    df_long = df_part.melt(
        id_vars=id_cols + meta_turnout_cols,
        value_vars=vote_cols,
        var_name="party_raw",
        value_name="votes"
    )

    df_long["party"] = df_long["party_raw"].apply(
        lambda x: clean_party_name(x, prefix)
    )

    df_long["election_type"] = election_type

    df_long["votes"] = clean_numeric_series(df_long["votes"])

    df_long = df_long.drop(columns=["party_raw"])

    # จัดลำดับ column ให้อ่านง่าย
    front_cols = ["election_type"] + id_cols + meta_turnout_cols + ["party", "votes"]
    front_cols = [col for col in front_cols if col in df_long.columns]

    df_long = df_long[front_cols]

    return df_long, vote_cols, meta_turnout_cols


# ============================================================
# 10) Make normal summary table
# ============================================================

def make_summary_table(df_long):
    """
    รวมคะแนนตามประเภทเลือกตั้งและพรรค
    """

    summary = (
        df_long
        .groupby(["election_type", "party"], as_index=False)["votes"]
        .sum()
        .sort_values("votes", ascending=False)
        .reset_index(drop=True)
    )

    return summary


# ============================================================
# 11) Make wide summary by party and subdistrict
# ============================================================

def find_subdistrict_col(df):
    """
    หา column สำหรับแยก Subdistrict / ตำบล

    ลำดับการหา:
    1) subdistrict
    2) subdistrict_name
    3) tambon
    4) ตำบล

    ถ้าไม่มีจริง ๆ จะ error เพื่อกันไม่ให้สรุปผิดพื้นที่
    """

    candidates = [
        "subdistrict",
        "subdistrict_name",
        "tambon",
        "ตำบล",
    ]

    for col in candidates:
        if col in df.columns:
            return col

    raise ValueError(
        "Cannot find subdistrict column. Expected one of: "
        "subdistrict, subdistrict_name, tambon, ตำบล"
    )


def make_party_subdistrict_wide_summary(df_long):
    """
    สร้างตารางแบบ:
    election_type | party | subdistrict_1 | subdistrict_2 | ... | total_votes

    โดย:
    - 1 row = 1 พรรค / 1 รายการ เช่น ก้าวไกล, เพื่อไทย, ไม่เลือกผู้ใด, บัตรดี, บัตรเสีย
    - column ตำบล = คะแนนของพรรคนั้นในแต่ละตำบล
    - total_votes = คะแนนรวมของพรรคนั้นทุกตำบล
    """

    df = df_long.copy()

    subdistrict_col = find_subdistrict_col(df)

    # กัน missing / ช่องว่าง
    df[subdistrict_col] = df[subdistrict_col].astype(str).str.strip()
    df[subdistrict_col] = df[subdistrict_col].replace(
        {
            "": "ไม่ระบุตำบล",
            "nan": "ไม่ระบุตำบล",
            "None": "ไม่ระบุตำบล",
        }
    )

    # รวมคะแนนก่อน เผื่อมีหลายหน่วยเลือกตั้งในตำบลเดียวกัน
    grouped = (
        df
        .groupby(["election_type", "party", subdistrict_col], as_index=False)["votes"]
        .sum()
    )

    # pivot ให้ตำบลกลายเป็น column
    wide = grouped.pivot_table(
        index=["election_type", "party"],
        columns=subdistrict_col,
        values="votes",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    wide.columns.name = None

    # หา columns ที่เป็นตำบล
    subdistrict_cols = [
        col for col in wide.columns
        if col not in ["election_type", "party"]
    ]

    # แปลงเป็น int
    for col in subdistrict_cols:
        wide[col] = pd.to_numeric(wide[col], errors="coerce").fillna(0).astype(int)

    # รวมคะแนนทุกตำบล
    wide["total_votes"] = wide[subdistrict_cols].sum(axis=1)

    # เรียงคะแนนรวมมากไปน้อย
    wide = (
        wide
        .sort_values(["election_type", "total_votes"], ascending=[True, False])
        .reset_index(drop=True)
    )

    return wide


# ============================================================
# 12) Main pipeline
# ============================================================

def main():
    # -------------------------
    # Download
    # -------------------------
    downloaded_file = download_from_google_drive(
        DRIVE_URL,
        OUTPUT_FILE
    )

    # -------------------------
    # Read
    # -------------------------
    df = read_election_file(downloaded_file)

    # -------------------------
    # Clean columns
    # -------------------------
    df = clean_column_names(df)

    print("\n==============================")
    print("Original dataframe")
    print("==============================")
    print("Shape:", df.shape)

    print("\nColumns:")
    for i, col in enumerate(df.columns):
        print(f"{i}: {col}")

    # -------------------------
    # Filter Chiang Mai District 7
    # -------------------------
    df_cm7 = filter_chiangmai_district7(df)

    print("\n==============================")
    print("Filtered Chiang Mai District 7")
    print("==============================")
    print("Shape:", df_cm7.shape)

    # -------------------------
    # Make constituency table
    # -------------------------
    df_constituency_long, constituency_vote_cols, constituency_meta_cols = make_vote_table(
        df_cm7,
        prefix=CONSTITUENCY_PREFIX,
        election_type="สส เขต"
    )

    # -------------------------
    # Make party-list table
    # -------------------------
    df_partylist_long, partylist_vote_cols, partylist_meta_cols = make_vote_table(
        df_cm7,
        prefix=PARTYLIST_PREFIX,
        election_type="บัญชีรายชื่อ"
    )

    # -------------------------
    # Make normal summaries
    # -------------------------
    summary_constituency = make_summary_table(df_constituency_long)
    summary_partylist = make_summary_table(df_partylist_long)

    # -------------------------
    # Make wide summaries by subdistrict
    # พรรค 1 แถว + แยกคะแนนตามตำบล
    # -------------------------
    summary_constituency_party_subdistrict_wide = make_party_subdistrict_wide_summary(
        df_constituency_long
    )

    summary_partylist_party_subdistrict_wide = make_party_subdistrict_wide_summary(
        df_partylist_long
    )

    summary_all_party_subdistrict_wide = pd.concat(
        [
            summary_constituency_party_subdistrict_wide,
            summary_partylist_party_subdistrict_wide
        ],
        ignore_index=True
    )

    # -------------------------
    # Print selected columns
    # -------------------------
    print("\n==============================")
    print("Selected columns")
    print("==============================")

    print("\nเขต vote columns:")
    for col in constituency_vote_cols:
        print("-", col)

    print("\nเขต metadata columns:")
    for col in constituency_meta_cols:
        print("-", col)

    print("\nบช vote columns:")
    for col in partylist_vote_cols:
        print("-", col)

    print("\nบช metadata columns:")
    for col in partylist_meta_cols:
        print("-", col)

    # -------------------------
    # Print result preview
    # -------------------------
    print("\n==============================")
    print("Result shapes")
    print("==============================")
    print("df_constituency_long:", df_constituency_long.shape)
    print("df_partylist_long:", df_partylist_long.shape)
    print("summary_constituency:", summary_constituency.shape)
    print("summary_partylist:", summary_partylist.shape)
    print("summary_constituency_party_subdistrict_wide:", summary_constituency_party_subdistrict_wide.shape)
    print("summary_partylist_party_subdistrict_wide:", summary_partylist_party_subdistrict_wide.shape)
    print("summary_all_party_subdistrict_wide:", summary_all_party_subdistrict_wide.shape)

    print("\n==============================")
    print("Constituency summary preview")
    print("==============================")
    print(summary_constituency.head(20))

    print("\n==============================")
    print("Party-list summary preview")
    print("==============================")
    print(summary_partylist.head(20))

    print("\n==============================")
    print("Constituency party subdistrict wide preview")
    print("==============================")
    print(summary_constituency_party_subdistrict_wide.head(20))

    print("\n==============================")
    print("Party-list party subdistrict wide preview")
    print("==============================")
    print(summary_partylist_party_subdistrict_wide.head(20))

    # -------------------------
    # Export
    # -------------------------
    df_cm7.to_csv(
        "chiangmai_district7_raw_filtered.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df_constituency_long.to_csv(
        "chiangmai_district7_constituency_long.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df_partylist_long.to_csv(
        "chiangmai_district7_partylist_long.csv",
        index=False,
        encoding="utf-8-sig"
    )

    summary_constituency.to_csv(
        "chiangmai_district7_constituency_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    summary_partylist.to_csv(
        "chiangmai_district7_partylist_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    summary_constituency_party_subdistrict_wide.to_csv(
        "chiangmai_district7_constituency_party_subdistrict_wide_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    summary_partylist_party_subdistrict_wide.to_csv(
        "chiangmai_district7_partylist_party_subdistrict_wide_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    summary_all_party_subdistrict_wide.to_csv(
        "chiangmai_district7_all_party_subdistrict_wide_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\n==============================")
    print("Export completed")
    print("==============================")
    print("1) chiangmai_district7_raw_filtered.csv")
    print("2) chiangmai_district7_constituency_long.csv")
    print("3) chiangmai_district7_partylist_long.csv")
    print("4) chiangmai_district7_constituency_summary.csv")
    print("5) chiangmai_district7_partylist_summary.csv")
    print("6) chiangmai_district7_constituency_party_subdistrict_wide_summary.csv")
    print("7) chiangmai_district7_partylist_party_subdistrict_wide_summary.csv")
    print("8) chiangmai_district7_all_party_subdistrict_wide_summary.csv")

    return {
        "raw_filtered": df_cm7,
        "constituency_long": df_constituency_long,
        "partylist_long": df_partylist_long,
        "constituency_summary": summary_constituency,
        "partylist_summary": summary_partylist,
        "constituency_party_subdistrict_wide_summary": summary_constituency_party_subdistrict_wide,
        "partylist_party_subdistrict_wide_summary": summary_partylist_party_subdistrict_wide,
        "all_party_subdistrict_wide_summary": summary_all_party_subdistrict_wide,
    }


# ============================================================
# 13) Run
# ============================================================

if __name__ == "__main__":
    results = main()

Downloading...
From: https://drive.google.com/uc?id=1inCkmOw8YtzX6kRf5R-yrU3gSRlIh6ZM
To: /content/election_result_file
100%|██████████| 29.0M/29.0M [00:00<00:00, 133MB/s]


Downloaded as: election_result_file
File type detected: CSV, encoding=utf-8-sig

Original dataframe
Shape: (99147, 98)

Columns:
0: id
1: index_no
2: document
3: province
4: province_number
5: district
6: subdistrict
7: registrar
8: station_number
9: บช_ก้าวไกล
10: บช_ชาติพัฒนากล้า
11: บช_ชาติไทยพัฒนา
12: บช_บัตรเสีย
13: บช_ประชาชาติ
14: บช_ประชาธิปัตย์
15: บช_ผู้มาใช้สิทธิ์
16: บช_ผู้มีสิทธิ์
17: บช_พลังประชารัฐ
18: บช_ภูมิใจไทย
19: บช_รวมไทยสร้างชาติ
20: บช_เพื่อไทย
21: บช_เสรีรวมไทย
22: บช_ไทยสร้างไทย
23: บช_ไม่เลือกผู้ใด
24: เขต_กรีน
25: เขต_ก้าวไกล
26: เขต_ครูไทยเพื่อประชาช
27: เขต_คลองไทย
28: เขต_ความหวังใหม่
29: เขต_ชาติพัฒนากล้า
30: เขต_ชาติรุ่งเรือง
31: เขต_ชาติไทยพัฒนา
32: เขต_ช่วยชาติ
33: เขต_ถิ่นกาขาวชาววิไล
34: เขต_ทางเลือกใหม่
35: เขต_ท้องที่ไทย
36: เขต_บัตรเสีย
37: เขต_ประชากรไทย
38: เขต_ประชาชาติ
39: เขต_ประชาธิปัตย์
40: เขต_ประชาธิปไตยใหม่
41: เขต_ประชาภิวัฒน์
42: เขต_ประชาสามัคคี
43: เขต_ประชาไทย
44: เขต_ผู้มาใช้สิทธิ์
45: เขต_ผู้มีสิทธิ์
46: เขต_พลัง
47: เขต_พลังธรรม